# Procesamiento del Lenguaje Natural (PLN)

# Question Answering (QA) utilizando Transformers

---

## Universidad de El Salvador

**Facultad de Ingeniería y Arquitectura**

**Escuela de Ingeniería de Sistemas Informáticos**

**Curso de Especialización**

### Objetivos

Al finalizar esta práctica el estudiante será capaz de:

- Comprender el funcionamiento de los sistemas Question Answering.
- Diferenciar los distintos tipos de modelos QA.
- Utilizar modelos preentrenados de Hugging Face.
- Implementar sistemas de preguntas y respuestas en español.
- Interpretar la confianza de las respuestas generadas.
- Analizar las limitaciones de los modelos actuales.

---

### Requisitos

- Python 3.10+
- Google Colab/ VS code / Spyder
- Transformers
- PyTorch

# Introducción

Question Answering (QA) es una tarea fundamental del Procesamiento del Lenguaje Natural cuyo objetivo consiste en responder automáticamente preguntas formuladas en lenguaje natural utilizando la información contenida en un texto.

A diferencia de un buscador tradicional, un sistema QA no devuelve documentos completos, sino la respuesta específica que responde a la pregunta.

Actualmente esta tecnología es utilizada por:

- ChatGPT
- Gemini
- Claude
- Copilot
- Alexa
- Siri
- Motores de búsqueda inteligentes

Los modelos modernos utilizan arquitecturas Transformer entrenadas con millones de ejemplos.

# Durante esta práctica aprenderemos a:

- instalar Transformers
- cargar modelos preentrenados
- utilizar Tokenizers
- construir un Pipeline de QA
- responder preguntas en español
- analizar la confianza de las respuestas
- trabajar con múltiples respuestas (`top_k`)
- detectar preguntas sin respuesta

# Question Answering

Un sistema Question Answering recibe dos entradas:

1. Un contexto
2. Una pregunta

y produce como salida:

- respuesta
- nivel de confianza
- posición dentro del texto

Ejemplo

Contexto:

> El Sistema Solar se formó hace aproximadamente 4.6 mil millones de años.

Pregunta:

> ¿Cuándo se formó el Sistema Solar?

Respuesta:

> 4.6 mil millones de años

# Tipos de Question Answering

Existen cuatro categorías principales.

## 1. Extractivo

Extrae literalmente un fragmento del contexto.

Ejemplo

Contexto

> El Sol es una estrella.

Pregunta

> ¿Qué es el Sol?

Respuesta

> una estrella

---

## 2. Abstractive

Genera una respuesta utilizando un modelo generativo.

---

## 3. Open Domain

Puede responder preguntas sobre cualquier tema.

---

## 4. Closed Domain

Está especializado en un dominio específico.

# Arquitectura Transformer

Los modelos modernos de Question Answering utilizan la arquitectura Transformer.

Componentes principales

- Embeddings
- Positional Encoding
- Multi-Head Attention
- Feed Forward Network
- Encoder
- Decoder

Para QA extractivo normalmente solo se utiliza el Encoder.

# Modelos para español

En esta práctica utilizaremos modelos entrenados específicamente para español.

Algunos modelos disponibles son:

|Modelo|Idioma|
|--------|--------|
|PlanTL-GOB-ES/roberta-base-bne-sqac|Español|
|Recognai/bert-base-spanish-wwm-cased-xquad|Español|
|mrm8488/bert-spanish-cased-finetuned-spa-squad2-es|Español|

Todos ellos fueron entrenados para responder preguntas utilizando texto en español.

In [1]:
# =====================================================
# INSTALACIÓN DE LIBRERÍAS
# =====================================================

!pip install -q transformers datasets sentencepiece accelerate

In [2]:
# =====================================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================================

import torch
import numpy as np
import pandas as pd

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

print("PyTorch:", torch.__version__)

PyTorch: 2.11.0+cpu


# Explicación de las librerías

## Transformers

Permite utilizar modelos de lenguaje desarrollados por Hugging Face.

## AutoTokenizer

Convierte texto en tokens.

## AutoModelForQuestionAnswering

Carga un modelo especializado en Question Answering.

## Pipeline

Simplifica el uso del modelo mediante una única función.

In [3]:
# =====================================================
# VERIFICAR GPU
# =====================================================

if torch.cuda.is_available():
    print("GPU disponible")
    print(torch.cuda.get_device_name(0))
else:
    print("Se utilizará CPU")

Se utilizará CPU


# ¿Qué haremos a continuación?

En el siguiente bloque aprenderemos a:

- cargar un modelo de Question Answering
- crear el Pipeline
- analizar el Tokenizer
- construir nuestra primera función para responder preguntas

# Carga del modelo de Question Answering

Los modelos de Question Answering se entrenan para localizar la respuesta a una pregunta dentro de un contexto.

El proceso consta de cuatro pasos:

1. Cargar el Tokenizer.
2. Cargar el modelo entrenado.
3. Construir un Pipeline.
4. Enviar preguntas y obtener respuestas.

Durante esta práctica utilizaremos un modelo entrenado para español.

# ¿Qué modelo utilizaremos?

Utilizaremos el modelo:

**PlanTL-GOB-ES/roberta-base-bne-sqac**

Características

- Arquitectura: RoBERTa Base
- Idioma: Español
- Tipo: Question Answering Extractivo
- Entrenado sobre SQAC (Spanish Question Answering Corpus)

Ventajas

- Excelente desempeño en español.
- Alta precisión.
- Compatible con Hugging Face.

In [4]:
from huggingface_hub import list_models

# Buscar modelos de PlanTL-GOB-ES
models = list_models(author="PlanTL-GOB-ES")
print("Modelos disponibles de PlanTL-GOB-ES:")
for model in models:
    print(f"  - {model.modelId}")

Modelos disponibles de PlanTL-GOB-ES:
  - PlanTL-GOB-ES/RoBERTalex
  - PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
  - PlanTL-GOB-ES/roberta-base-biomedical-es
  - PlanTL-GOB-ES/roberta-base-ca
  - PlanTL-GOB-ES/bsc-bio-ehr-es-pharmaconer
  - PlanTL-GOB-ES/bsc-bio-ehr-es-cantemist
  - PlanTL-GOB-ES/bsc-bio-es
  - PlanTL-GOB-ES/bsc-bio-ehr-es
  - PlanTL-GOB-ES/longformer-base-4096-biomedical-clinical-es
  - PlanTL-GOB-ES/es_pharmaconer_ner_trf
  - PlanTL-GOB-ES/es_cantemist_ner_trf
  - PlanTL-GOB-ES/es_bsc_demo_trf
  - PlanTL-GOB-ES/roberta-base-es-wikicat-es
  - PlanTL-GOB-ES/mt-plantl-es-ca
  - PlanTL-GOB-ES/mt-plantl-es-gl
  - PlanTL-GOB-ES/es_bsc_demo_md
  - PlanTL-GOB-ES/ca_anonimization_core_lg
  - PlanTL-GOB-ES/es_anonimization_core_lg
  - PlanTL-GOB-ES/Controversy-Prediction
  - PlanTL-GOB-ES/roberta-large-bne
  - PlanTL-GOB-ES/roberta-base-bne
  - PlanTL-GOB-ES/longformer-base-4096-bne-es
  - PlanTL-GOB-ES/gpt2-base-bne
  - PlanTL-GOB-ES/gpt2-large-bne


# ¿Qué es un Tokenizer?

Los modelos Transformer no trabajan directamente con palabras.

Antes de procesar un texto, éste debe convertirse en números.

El Tokenizer realiza tres tareas:

- dividir el texto en tokens
- asignar un identificador a cada token
- preparar la entrada para la red neuronal

Ejemplo

Texto

```
La inteligencia artificial está transformando el mundo.
```

Tokens

```
["La",
"inteligencia",
"artificial",
"está",
"transformando",
"el",
"mundo"]
```

# Nota de implementación

La guía define la función `responder_pregunta()` **dos veces**, con el orden de los
argumentos invertido entre una definición y otra. Como Python conserva la última
definición ejecutada, el resultado depende del orden en que se corran las celdas:
pregunta y contexto pueden intercambiarse silenciosamente y el modelo responde sobre
el texto equivocado sin lanzar ningún error.

Adicionalmente, la selección del span mediante `argmax` independiente sobre los logits
de inicio y de fin presenta tres defectos:

| Defecto | Consecuencia |
|---|---|
| No se enmascaran los tokens de la pregunta | La respuesta puede extraerse de la propia pregunta o de `[CLS]` / `[SEP]` |
| No se exige `fin >= inicio` | Se producen spans invertidos que devuelven texto vacío |
| No se acotan las posiciones a caracteres | `start` y `end` indexan tokens, no el contexto original |

Para resolverlo se consolida **una única implementación** en el módulo
`src/funciones_qa.py`, que evalúa pares `(inicio, fin)` válidos y los puntúa como
`P(inicio) · P(fin)`. Todo el laboratorio —QA básico, comparación de modelos y sistema
RAG— utiliza ese mismo camino de código, de modo que las diferencias observadas entre
experimentos sean atribuibles al modelo o al contexto, nunca a la implementación.

In [ ]:
# =====================================================
# ESTRUCTURA DEL PROYECTO
# =====================================================

import os

for carpeta in ["src", "data", "results", "results/graficos", "docs"]:
    os.makedirs(carpeta, exist_ok=True)

print("Directorios listos:", os.listdir("."))

In [ ]:
%%writefile src/funciones_qa.py
"""
funciones_qa.py
===============

Funciones reutilizables para el Laboratorio No. 2 — Question Answering extractivo
en español con modelos Transformer.

Universidad de El Salvador — Escuela de Ingeniería de Sistemas Informáticos
Autor: Mahalaleel Villalta Martínez

Este módulo centraliza la lógica de inferencia QA para que el notebook, la
comparación de modelos y el sistema RAG utilicen exactamente el mismo camino
de código. Así cualquier diferencia observada entre experimentos es atribuible
al modelo o al contexto, nunca a la implementación.
"""

from __future__ import annotations

import time
from typing import Any

import numpy as np
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

__all__ = [
    "dispositivo",
    "ModeloQA",
    "decodificar_spans",
    "responder_seguro",
    "medir_tiempo",
]


# ---------------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------------

def dispositivo() -> torch.device:
    """Devuelve GPU si está disponible, CPU en caso contrario."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _softmax(x: np.ndarray) -> np.ndarray:
    """Softmax numéricamente estable."""
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()


# ---------------------------------------------------------------------------
# Decodificación de spans
# ---------------------------------------------------------------------------

def decodificar_spans(
    start_logits: np.ndarray,
    end_logits: np.ndarray,
    offsets: list,
    sequence_ids: list,
    contexto: str,
    top_k: int = 1,
    max_answer_len: int = 50,
    n_mejores: int = 20,
) -> list[dict[str, Any]]:
    """
    Convierte los logits del modelo en respuestas de texto.

    A diferencia de tomar simplemente ``argmax`` sobre cada vector de logits,
    esta función:

    1. Enmascara todo token que no pertenezca al contexto, de modo que la
       respuesta nunca pueda extraerse de la pregunta ni de los tokens
       especiales ([CLS], [SEP], <s>, </s>).
    2. Evalúa **pares** (inicio, fin) válidos exigiendo ``fin >= inicio`` y una
       longitud máxima de respuesta. El argmax independiente puede producir un
       fin anterior al inicio, devolviendo texto vacío o invertido.
    3. Puntúa cada par como ``P(inicio) * P(fin)``, que es la probabilidad
       conjunta del span bajo la suposición de independencia usada por SQuAD.
    4. Traduce posiciones de token a posiciones de carácter mediante el
       ``offset_mapping``, por lo que ``start`` y ``end`` indexan el contexto
       original y ``contexto[start:end]`` reproduce la respuesta exactamente.

    Parámetros
    ----------
    start_logits, end_logits : np.ndarray
        Logits de inicio y fin, de forma ``(n_tokens,)``.
    offsets : list
        ``offset_mapping`` del tokenizer: pares (inicio_char, fin_char).
    sequence_ids : list
        Salida de ``encoding.sequence_ids()``. Vale 0 para la pregunta,
        1 para el contexto y ``None`` para tokens especiales.
    contexto : str
        Texto original sobre el que se indexan los offsets.
    top_k : int
        Número de respuestas candidatas a devolver.
    max_answer_len : int
        Longitud máxima de la respuesta, en tokens.
    n_mejores : int
        Cuántas posiciones de inicio y de fin se consideran antes de
        emparejarlas. Controla el costo del producto cartesiano.

    Retorna
    -------
    list[dict]
        Respuestas ordenadas por score descendente. Cada una contiene
        ``answer``, ``score``, ``start`` y ``end``.
    """
    start_logits = np.asarray(start_logits, dtype=np.float64)
    end_logits = np.asarray(end_logits, dtype=np.float64)

    # 1. Enmascarar todo lo que no sea contexto
    fuera_de_contexto = np.array([sid != 1 for sid in sequence_ids])
    start_logits = np.where(fuera_de_contexto, -1e9, start_logits)
    end_logits = np.where(fuera_de_contexto, -1e9, end_logits)

    start_probs = _softmax(start_logits)
    end_probs = _softmax(end_logits)

    # 2. Emparejar candidatos válidos
    mejores_inicio = np.argsort(start_probs)[::-1][:n_mejores]
    mejores_fin = np.argsort(end_probs)[::-1][:n_mejores]

    candidatos: dict[str, dict[str, Any]] = {}

    for i in mejores_inicio:
        for j in mejores_fin:
            if j < i:
                continue
            if (j - i + 1) > max_answer_len:
                continue
            if fuera_de_contexto[i] or fuera_de_contexto[j]:
                continue

            char_inicio = int(offsets[i][0])
            char_fin = int(offsets[j][1])
            if char_fin <= char_inicio:
                continue

            texto = contexto[char_inicio:char_fin].strip()
            if not texto:
                continue

            score = float(start_probs[i] * end_probs[j])

            # 3. Conservar solo la mejor aparición de cada texto distinto
            previo = candidatos.get(texto)
            if previo is None or score > previo["score"]:
                candidatos[texto] = {
                    "answer": texto,
                    "score": score,
                    "start": char_inicio,
                    "end": char_fin,
                }

    ordenados = sorted(candidatos.values(), key=lambda r: r["score"], reverse=True)

    if not ordenados:
        return [{"answer": "", "score": 0.0, "start": 0, "end": 0}]

    return ordenados[:top_k]


# ---------------------------------------------------------------------------
# Modelo
# ---------------------------------------------------------------------------

class ModeloQA:
    """
    Envoltorio sobre un modelo de Question Answering extractivo de Hugging Face.

    Ejemplo
    -------
    >>> qa = ModeloQA("mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")
    >>> qa.responder("La UES fue fundada en 1841.", "¿Cuándo fue fundada la UES?")
    [{'answer': '1841', 'score': 0.97, 'start': 25, 'end': 29}]
    """

    def __init__(self, nombre_modelo: str, etiqueta: str | None = None, device=None):
        self.nombre_modelo = nombre_modelo
        self.etiqueta = etiqueta or nombre_modelo.split("/")[-1]
        self.device = device or dispositivo()

        self.tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
        self.model = AutoModelForQuestionAnswering.from_pretrained(nombre_modelo)
        self.model.to(self.device)
        self.model.eval()

    # -- Introspección ------------------------------------------------------

    @property
    def n_parametros(self) -> int:
        return sum(p.numel() for p in self.model.parameters())

    @property
    def max_tokens(self) -> int:
        return self.tokenizer.model_max_length

    def describir(self) -> dict[str, Any]:
        return {
            "Etiqueta": self.etiqueta,
            "Modelo": self.nombre_modelo,
            "Arquitectura": self.model.config.architectures[0],
            "Parámetros (M)": round(self.n_parametros / 1e6, 1),
            "Tokenizer": type(self.tokenizer).__name__,
            "Vocabulario": self.tokenizer.vocab_size,
            "Dispositivo": str(self.device),
        }

    # -- Inferencia ---------------------------------------------------------

    def responder(
        self,
        contexto: str,
        pregunta: str,
        top_k: int = 1,
        max_answer_len: int = 50,
        max_length: int = 384,
    ) -> list[dict[str, Any]]:
        """Responde una pregunta sobre un contexto. Ver ``decodificar_spans``."""
        codificacion = self.tokenizer(
            pregunta,
            contexto,
            return_tensors="pt",
            truncation="only_second",   # nunca recortar la pregunta
            max_length=max_length,
            return_offsets_mapping=True,
        )

        offsets = codificacion.pop("offset_mapping")[0].tolist()
        seq_ids = codificacion.sequence_ids(0)

        entradas = {k: v.to(self.device) for k, v in codificacion.items()}

        with torch.no_grad():
            salida = self.model(**entradas)

        return decodificar_spans(
            start_logits=salida.start_logits[0].detach().cpu().numpy(),
            end_logits=salida.end_logits[0].detach().cpu().numpy(),
            offsets=offsets,
            sequence_ids=seq_ids,
            contexto=contexto,
            top_k=top_k,
            max_answer_len=max_answer_len,
        )

    def responder_con_tiempo(self, contexto: str, pregunta: str, **kwargs):
        """Igual que ``responder``, pero devuelve también el tiempo en segundos."""
        inicio = time.perf_counter()
        respuestas = self.responder(contexto, pregunta, **kwargs)
        return respuestas, time.perf_counter() - inicio

    def __call__(self, contexto: str, pregunta: str, **kwargs):
        return self.responder(contexto, pregunta, **kwargs)

    def __repr__(self) -> str:
        return f"ModeloQA({self.etiqueta!r}, device={self.device})"


# ---------------------------------------------------------------------------
# Utilidades de alto nivel
# ---------------------------------------------------------------------------

def responder_seguro(
    modelo: ModeloQA,
    contexto: str,
    pregunta: str,
    umbral: float = 0.40,
) -> dict[str, Any]:
    """
    Responde aplicando un umbral de confianza.

    Un modelo extractivo siempre devuelve algún fragmento del contexto, incluso
    cuando la respuesta no está presente. El umbral permite marcar esos casos
    en lugar de propagar una respuesta sin fundamento.
    """
    respuesta = modelo.responder(contexto, pregunta, top_k=1)[0]
    confiable = respuesta["score"] >= umbral

    return {
        **respuesta,
        "pregunta": pregunta,
        "confiable": confiable,
        "veredicto": "Respuesta aceptada" if confiable
                     else "Confianza insuficiente — requiere verificación manual",
    }


def medir_tiempo(funcion, *args, repeticiones: int = 3, **kwargs):
    """
    Ejecuta una función varias veces y devuelve (resultado, tiempo_medio).

    Se descarta la primera ejecución porque incluye la inicialización de kernels
    CUDA y el calentamiento de cachés, que distorsionan la medición.
    """
    funcion(*args, **kwargs)  # calentamiento

    tiempos = []
    resultado = None
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = funcion(*args, **kwargs)
        tiempos.append(time.perf_counter() - inicio)

    return resultado, float(np.mean(tiempos))


In [ ]:
# =====================================================
# CARGA DEL MÓDULO
# =====================================================

import sys
sys.path.append(".")

from src.funciones_qa import (
    ModeloQA,
    responder_seguro,
    medir_tiempo,
    dispositivo,
)

print("Dispositivo:", dispositivo())

In [ ]:
# =====================================================
# MODELOS QA EN ESPAÑOL
# =====================================================
# Verificados contra la API de Hugging Face en agosto de 2026.
#
# Los tres modelos citados en la guía original ya no son utilizables:
#   - PlanTL-GOB-ES/roberta-base-bne-sqac          -> retirado del Hub (404)
#   - Recognai/bert-base-spanish-wwm-cased-xquad   -> no existe; el modelo de
#                                                     Recognai es de clasificación
#                                                     zero-shot, no de QA
#   - mrm8488/bert-spanish-cased-finetuned-spa-...  -> 404; el identificador real
#                                                     incluye "wwm-cased"
#
# Se sustituyen por tres modelos que sí varían en arquitectura, tamaño y
# tokenizador, de modo que la comparación del Bloque 9 resulte significativa.

MODELOS = {
    "DistilBETO-SQuAD2-es":
        "mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",

    "BETO-SQuAD2-es":
        "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",

    "XLM-R-SQuAD2":
        "deepset/xlm-roberta-base-squad2",
}

MODELO_PRINCIPAL = "BETO-SQuAD2-es"

for etiqueta, ruta in MODELOS.items():
    print(f"{etiqueta:24s} -> {ruta}")

In [ ]:
# =====================================================
# CARGA DEL MODELO PRINCIPAL
# =====================================================

import pandas as pd

qa = ModeloQA(
    MODELOS[MODELO_PRINCIPAL],
    etiqueta=MODELO_PRINCIPAL
)

pd.DataFrame([qa.describir()]).T.rename(columns={0: "Valor"})

In [ ]:
# =====================================================
# FUNCIÓN responder_pregunta()
# =====================================================
# Definición ÚNICA para todo el notebook.
# Orden de argumentos: (contexto, pregunta).

def responder_pregunta(contexto, pregunta, top_k=1, **kwargs):
    """
    Responde una pregunta localizando el fragmento correspondiente
    dentro del contexto.

    Parámetros
    ----------
    contexto : str   Texto donde buscar la respuesta.
    pregunta : str   Pregunta formulada.
    top_k    : int   Número de respuestas candidatas a devolver.

    Retorna
    -------
    list[dict]  Respuestas ordenadas por score, con las claves
                answer, score, start y end.
    """
    return qa.responder(contexto, pregunta, top_k=top_k, **kwargs)


print(responder_pregunta.__doc__)

## Verificación de la implementación

Antes de utilizar la función en el resto del laboratorio se comprueban tres
propiedades que la implementación original no garantizaba.

In [ ]:
# =====================================================
# VERIFICACIÓN
# =====================================================

contexto_prueba = (
    "La Universidad de El Salvador fue fundada el 16 de febrero de 1841. "
    "Es la universidad pública más antigua del país y su campus central "
    "se ubica en San Salvador."
)

pruebas = []

# 1. start/end deben indexar el contexto original
r = responder_pregunta(contexto_prueba, "¿En qué año fue fundada la UES?")[0]
pruebas.append({
    "Verificación": "contexto[start:end] reproduce la respuesta",
    "Resultado": contexto_prueba[r["start"]:r["end"]] == r["answer"],
    "Evidencia": f"{r['answer']!r} en [{r['start']}:{r['end']}]",
})

# 2. La respuesta nunca debe provenir de la pregunta
r2 = responder_pregunta(contexto_prueba, "¿Cuál es la capital de Francia?")[0]
pruebas.append({
    "Verificación": "La respuesta procede del contexto, no de la pregunta",
    "Resultado": r2["answer"] in contexto_prueba,
    "Evidencia": f"{r2['answer']!r} (score {r2['score']:.4f})",
})

# 3. top_k debe devolver candidatos distintos y ordenados
rs = responder_pregunta(contexto_prueba, "¿Dónde se ubica el campus central?", top_k=3)
scores = [x["score"] for x in rs]
pruebas.append({
    "Verificación": "top_k devuelve candidatos únicos y ordenados",
    "Resultado": (len({x["answer"] for x in rs}) == len(rs)
                  and scores == sorted(scores, reverse=True)),
    "Evidencia": " | ".join(f"{x['answer']!r}" for x in rs),
})

pd.DataFrame(pruebas)

# ¿Qué hace el modelo?

El modelo recibe:

- una pregunta
- un contexto

Internamente calcula la probabilidad de que cada palabra del contexto sea el inicio y el final de la respuesta.

Finalmente devuelve:

- respuesta
- puntuación de confianza
- posición inicial
- posición final

# Pipeline de Hugging Face

Un Pipeline simplifica el uso del modelo.

Sin Pipeline habría que:

- tokenizar
- convertir tensores
- ejecutar el modelo
- calcular probabilidades
- decodificar la respuesta

El Pipeline realiza automáticamente todos esos pasos.

# ¿Qué devuelve un Pipeline QA?

Cada respuesta contiene:

- answer
- score
- start
- end

Ejemplo

```python
{
 'answer':'Ganímedes',
 'score':0.92,
 'start':215,
 'end':224
}
```

El atributo **score** representa el nivel de confianza del modelo.

# Resumen del Bloque

En este bloque aprendimos a:

- Cargar un modelo de Question Answering.
- Comprender el funcionamiento del Tokenizer.
- Construir un Pipeline de Hugging Face.
- Interpretar la estructura de una respuesta.
- Implementar una función reutilizable para responder preguntas.
- Obtener múltiples respuestas mediante el parámetro `top_k`.

En el siguiente bloque se trabajará con contextos más extensos (Sistema Solar, Inteligencia Artificial y Universidad de El Salvador), además de analizar el comportamiento del modelo ante diferentes tipos de preguntas y distintos niveles de confianza.

In [8]:
# =====================================================
# PROBAR LA FUNCIÓN
# =====================================================

contexto = """
La Universidad de El Salvador fue fundada en 1841.
Es la universidad pública más antigua del país.
Su campus central está ubicado en San Salvador.
"""

pregunta = "¿En qué año fue fundada la Universidad de El Salvador?"

respuesta = responder_pregunta(
    contexto,
    pregunta
)

respuesta

[{'answer': '1841', 'score': 0.9859557747840881, 'start': 22, 'end': 23},
 {'answer': '1841.', 'score': 0.002889536786824465, 'start': 22, 'end': 24},
 {'answer': '[CLS] ¿ en que ano fue fundada la universidad de el salvador? [SEP] la universidad de el salvador fue fundada en 1841',
  'score': 0.0028839248698204756,
  'start': 0,
  'end': 23}]

In [9]:
# =====================================================
# MOSTRAR LA MEJOR RESPUESTA
# =====================================================

print("="*50)

print("Pregunta")
print(pregunta)

print("\nRespuesta")
print(respuesta[0]["answer"])

print("\nConfianza")
print(round(respuesta[0]["score"],4))

Pregunta
¿En qué año fue fundada la Universidad de El Salvador?

Respuesta
1841

Confianza
0.986


In [10]:
# =====================================================
# MOSTRAR MÚLTIPLES RESPUESTAS
# =====================================================

respuestas = responder_pregunta(
    contexto,
    pregunta,
    top_k=5
)

print("="*60)
print("RESPUESTAS CANDIDATAS")
print("="*60)

for i, r in enumerate(respuestas, start=1):
    print(f"\nRespuesta {i}")
    print(f"Texto      : {r['answer']}")
    print(f"Confianza  : {r['score']:.4f}")

RESPUESTAS CANDIDATAS

Respuesta 1
Texto      : 1841
Confianza  : 0.9860

Respuesta 2
Texto      : 1841.
Confianza  : 0.0029

Respuesta 3
Texto      : [CLS] ¿ en que ano fue fundada la universidad de el salvador? [SEP] la universidad de el salvador fue fundada en 1841
Confianza  : 0.0029

Respuesta 4
Texto      : 184
Confianza  : 0.0019

Respuesta 5
Texto      : en 1841
Confianza  : 0.0010


# Resumen del Bloque

En este bloque aprendimos a:

- Cargar un modelo de Question Answering.
- Comprender el funcionamiento del Tokenizer.
- Construir un Pipeline de Hugging Face.
- Interpretar la estructura de una respuesta.
- Implementar una función reutilizable para responder preguntas.
- Obtener múltiples respuestas mediante el parámetro `top_k`.

En el siguiente bloque se trabajará con contextos más extensos (Sistema Solar, Inteligencia Artificial y Universidad de El Salvador), además de analizar el comportamiento del modelo ante diferentes tipos de preguntas y distintos niveles de confianza.

# Ejemplo 1. Sistema Solar

En este primer ejemplo utilizaremos un contexto relativamente largo.

El objetivo es observar cómo el modelo identifica automáticamente el fragmento del texto que responde a una pregunta determinada.

Durante este ejemplo analizaremos:

- calidad de la respuesta
- score de confianza
- múltiples respuestas (`top_k`)
- posición de la respuesta dentro del texto

In [11]:
# =====================================================
# CONTEXTO 1
# SISTEMA SOLAR
# =====================================================

contexto_solar = """
El Sistema Solar es el conjunto de cuerpos celestes que giran alrededor del Sol.

Se formó hace aproximadamente 4.6 mil millones de años debido al colapso gravitacional de una nube molecular.

Está compuesto por ocho planetas principales:

Mercurio,
Venus,
Tierra,
Marte,
Júpiter,
Saturno,
Urano
y Neptuno.

Júpiter es el planeta más grande del Sistema Solar.

Ganímedes, una luna de Júpiter, es el satélite natural más grande del Sistema Solar.

La mayor parte de la masa del Sistema Solar corresponde al Sol.
"""

## Pregunta 1

**¿Cuándo se formó el Sistema Solar?**

El modelo deberá localizar la fecha dentro del contexto.

In [12]:
pregunta = "¿Cuándo se formó el Sistema Solar?"

respuesta = responder_pregunta(
    contexto_solar,
    pregunta,
    top_k=3
)

respuesta

[{'answer': 'hace aproximadamente 4. 6 mil millones de anos',
  'score': 0.4050234854221344,
  'start': 31,
  'end': 40},
 {'answer': '4. 6 mil millones de anos',
  'score': 0.3638102114200592,
  'start': 33,
  'end': 40},
 {'answer': 'aproximadamente 4. 6 mil millones de anos',
  'score': 0.15057606995105743,
  'start': 32,
  'end': 40}]

In [13]:
print("="*60)

print("Pregunta")
print(pregunta)

print("\nRespuesta")

for i,r in enumerate(respuesta):

    print(f"\nRespuesta {i+1}")

    print("Texto      :",r["answer"])

    print("Confianza  :",round(r["score"],4))

    print("Inicio     :",r["start"])

    print("Fin        :",r["end"])

Pregunta
¿Cuándo se formó el Sistema Solar?

Respuesta

Respuesta 1
Texto      : hace aproximadamente 4. 6 mil millones de anos
Confianza  : 0.405
Inicio     : 31
Fin        : 40

Respuesta 2
Texto      : 4. 6 mil millones de anos
Confianza  : 0.3638
Inicio     : 33
Fin        : 40

Respuesta 3
Texto      : aproximadamente 4. 6 mil millones de anos
Confianza  : 0.1506
Inicio     : 32
Fin        : 40


## Pregunta 2

¿Cuál es el planeta más grande del Sistema Solar?

In [14]:
pregunta = "¿Cuál es el planeta más grande del Sistema Solar?"

respuesta = responder_pregunta(
    contexto_solar,
    pregunta,
    top_k=5
)

for r in respuesta:

    print(r)

{'answer': 'jupiter', 'score': 0.6398580074310303, 'start': 86, 'end': 88}
{'answer': 'jupiter es', 'score': 0.043491262942552567, 'start': 86, 'end': 89}
{'answer': 'neptuno. jupiter', 'score': 0.030204560607671738, 'start': 82, 'end': 88}
{'answer': 'jupiter, saturno, urano y neptuno. jupiter', 'score': 0.023761238902807236, 'start': 71, 'end': 88}
{'answer': 'mercurio, venus, tierra, marte, jupiter, saturno, urano y neptuno. jupiter', 'score': 0.006902943830937147, 'start': 61, 'end': 88}


## Pregunta 3

¿Cuál es el satélite natural más grande?

In [15]:
pregunta = "¿Cuál es el satélite natural más grande?"

respuesta = responder_pregunta(
    contexto_solar,
    pregunta,
    top_k=5
)

for r in respuesta:

    print(r)

{'answer': 'ganimedes', 'score': 0.8539800643920898, 'start': 98, 'end': 100}
{'answer': 'ganimedes,', 'score': 0.012843425385653973, 'start': 98, 'end': 101}
{'answer': 'ganimedes, una luna de jupiter', 'score': 0.009893633425235748, 'start': 98, 'end': 107}
{'answer': 'ganimedes, una luna de jupiter, es el satelite natural mas grande del sistema solar', 'score': 0.0051299892365932465, 'start': 98, 'end': 119}
{'answer': 'jupiter es el planeta mas grande del sistema solar. ganimedes', 'score': 0.0016320743598043919, 'start': 86, 'end': 100}


## Interpretación

Observe que el modelo no genera texto nuevo.

Simplemente extrae un fragmento del contexto.

Por ello este tipo de modelo recibe el nombre de **Question Answering Extractivo**.

# Ejemplo 2. Inteligencia Artificial

In [16]:
contexto_ia = """
La Inteligencia Artificial es una rama de la informática que desarrolla sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.

El término Inteligencia Artificial fue propuesto por John McCarthy durante la Conferencia de Dartmouth en 1956.

Actualmente la IA tiene aplicaciones en medicina, educación, agricultura, robótica, industria, transporte y finanzas.

Los modelos modernos utilizan técnicas de Machine Learning y Deep Learning.
"""

In [17]:
preguntas = [

"¿Quién propuso el término Inteligencia Artificial?",

"¿En qué año fue propuesta?",

"¿Qué técnicas utilizan los modelos modernos?",

"¿En qué áreas se aplica la IA?"
]

for pregunta in preguntas:

    print("="*70)

    print(pregunta)

    respuesta = responder_pregunta(
        contexto_ia,
        pregunta,
        top_k=3
    )

    for r in respuesta:

        print(f"Respuesta : {r['answer']}")

        print(f"Score     : {r['score']:.4f}")

        print()

¿Quién propuso el término Inteligencia Artificial?
Respuesta : john mccarthy
Score     : 0.9712

Respuesta : mccarthy
Score     : 0.0058

Respuesta : el termino inteligencia artificial fue propuesto por john mccarthy
Score     : 0.0029

¿En qué año fue propuesta?
Respuesta : 1956
Score     : 0.8262

Respuesta : 1956.
Score     : 0.0051

Respuesta : en 1956
Score     : 0.0029

¿Qué técnicas utilizan los modelos modernos?
Respuesta : machine learning y
Score     : 0.3138

Respuesta : machine learning y deep learning
Score     : 0.2691

Respuesta : tecnicas de machine learning y
Score     : 0.0652

¿En qué áreas se aplica la IA?
Respuesta : medicina, educacion, agricultura, robotica, industria, transporte y finanzas
Score     : 0.8559

Respuesta : medicina, educacion, agricultura, robotica, industria, transporte y finanzas.
Score     : 0.0248

Respuesta : educacion, agricultura, robotica, industria, transporte y finanzas
Score     : 0.0109



## Analizando la confianza

No todas las respuestas presentan el mismo nivel de confianza.

Factores que influyen:

- longitud del contexto

- claridad de la pregunta

- ambigüedad

- calidad del corpus de entrenamiento

- complejidad del lenguaje

In [18]:
print("="*60)

pregunta="¿Quién inventó Internet?"

respuesta=responder_pregunta(
    contexto_ia,
    pregunta,
    top_k=5
)

for r in respuesta:

    print(r)

{'answer': '[CLS] ¿ quien invento internet? [SEP] la inteligencia artificial', 'score': 4.493249434744939e-05, 'start': 0, 'end': 9}
{'answer': '[CLS] ¿ quien invento internet? [SEP] la inteligencia artificial es una rama de la informatica que desarrolla sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.', 'score': 2.714494985411875e-05, 'start': 0, 'end': 29}
{'answer': 'john mccarthy', 'score': 6.737428179803828e-07, 'start': 37, 'end': 42}
{'answer': 'john mccarthy durante la conferencia de dartmouth en 1956', 'score': 2.5070414721994894e-07, 'start': 37, 'end': 51}
{'answer': '1956', 'score': 6.691215759246916e-08, 'start': 51, 'end': 51}


## Discusión

Observe que la pregunta anterior no puede responderse utilizando el contexto.

Analice:

- ¿El modelo devuelve una respuesta?

- ¿La respuesta tiene sentido?

- ¿Cómo cambia el score?

- ¿Por qué ocurre este comportamiento?

In [19]:
pregunta="¿Cuál es la capital de Francia?"

respuesta=responder_pregunta(
    contexto_ia,
    pregunta,
    top_k=5
)

for r in respuesta:

    print(r)

{'answer': '[CLS] ¿ cual es la capital de francia', 'score': 0.004032142460346222, 'start': 0, 'end': 8}
{'answer': '[CLS] ¿ cual es la capital de francia? [SEP] la inteligencia artificial', 'score': 0.002792918123304844, 'start': 0, 'end': 13}
{'answer': '[CLS] ¿ cual es la capital de', 'score': 0.0023545760195702314, 'start': 0, 'end': 6}
{'answer': '1956', 'score': 6.0344838857417926e-05, 'start': 55, 'end': 55}
{'answer': '1956. actualmente la ia', 'score': 5.8406843891134486e-05, 'start': 55, 'end': 60}


# Actividad 1

Explique con sus propias palabras la diferencia entre:

- buscar información
- responder preguntas

¿Cuáles son las ventajas de un sistema QA sobre un buscador tradicional?

# Actividad 2

Construya un contexto de aproximadamente 200 palabras sobre un tema de su interés.

Posteriormente formule cinco preguntas cuya respuesta pueda encontrarse dentro del contexto.

Analice los resultados obtenidos.

# Resumen del bloque

En este bloque aprendimos a:

- utilizar el modelo con distintos contextos

- interpretar el score

- obtener múltiples respuestas

- comprender que un modelo extractivo únicamente puede responder utilizando información presente en el contexto

En el siguiente bloque estudiaremos cómo funciona el parámetro `top_k`, analizaremos preguntas sin respuesta y compararemos distintos modelos de Question Answering en español.

# Profundizando en Question Answering

Hasta ahora hemos utilizado el modelo como una "caja negra".

En este bloque analizaremos con mayor detalle:

- ¿Cómo selecciona una respuesta?
- ¿Por qué existen varias respuestas posibles?
- ¿Qué significa el score?
- ¿Qué ocurre cuando el contexto no contiene la respuesta?
- ¿Cómo influye la calidad del contexto?

# ¿Qué significa `top_k`?

El parámetro `top_k` indica cuántas respuestas candidatas devolverá el modelo.

Por ejemplo:

```python
top_k=1
```

devuelve únicamente la respuesta más probable.

Mientras que

```python
top_k=5
```

devuelve las cinco respuestas con mayor probabilidad.

Esto resulta útil para analizar el comportamiento del modelo y comparar distintas alternativas.

In [20]:
# =====================================================
# COMPARACIÓN DE top_k
# =====================================================

pregunta = "¿Cuál es el planeta más grande del Sistema Solar?"

for k in [1, 3, 5]:

    print("="*70)
    print(f"top_k = {k}")

    respuestas = responder_pregunta(
        contexto_solar,
        pregunta,
        top_k=k
    )

    for i, r in enumerate(respuestas, start=1):

        print(f"\nRespuesta {i}")

        print("Texto      :", r["answer"])

        print("Confianza  :", round(r["score"],4))

top_k = 1

Respuesta 1
Texto      : jupiter
Confianza  : 0.6399
top_k = 3

Respuesta 1
Texto      : jupiter
Confianza  : 0.6399

Respuesta 2
Texto      : jupiter es
Confianza  : 0.0435

Respuesta 3
Texto      : neptuno. jupiter
Confianza  : 0.0302
top_k = 5

Respuesta 1
Texto      : jupiter
Confianza  : 0.6399

Respuesta 2
Texto      : jupiter es
Confianza  : 0.0435

Respuesta 3
Texto      : neptuno. jupiter
Confianza  : 0.0302

Respuesta 4
Texto      : jupiter, saturno, urano y neptuno. jupiter
Confianza  : 0.0238

Respuesta 5
Texto      : mercurio, venus, tierra, marte, jupiter, saturno, urano y neptuno. jupiter
Confianza  : 0.0069


# Observación

Analice los resultados obtenidos.

Preguntas para discutir:

- ¿La mejor respuesta cambia?
- ¿Las respuestas adicionales son útiles?
- ¿Qué ocurre con los valores del score?

In [ ]:
# =====================================================
# MOSTRAR RESPUESTAS EN TABLA
# =====================================================

import pandas as pd

respuestas = responder_pregunta(
    contexto_solar,
    "¿Cuál es el planeta más grande del Sistema Solar?",
    top_k=5
)

tabla_topk = pd.DataFrame(respuestas)

tabla_topk

# Interpretando la tabla

Cada fila representa una respuesta candidata.

Las columnas significan:

- answer → respuesta encontrada.
- score → nivel de confianza.
- start → posición inicial.
- end → posición final.

In [ ]:
# =====================================================
# ORDENANDO RESPUESTAS
# =====================================================

tabla_topk = tabla_topk.sort_values(
    by="score",
    ascending=False
)

tabla_topk

# ¿Cómo interpretar el score?

No existe un valor mínimo universal.

Sin embargo, una guía práctica es:

|Score|Interpretación|
|------|--------------|
|>0.90|Excelente|
|0.80-0.90|Muy buena|
|0.60-0.79|Aceptable|
|0.40-0.59|Baja confianza|
|<0.40|Debe verificarse|

In [ ]:
# =====================================================
# VISUALIZAR SCORES
# =====================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

plt.bar(
    range(len(tabla_topk)),
    tabla_topk["score"]
)

plt.xticks(
    range(len(tabla_topk)),
    tabla_topk["answer"],
    rotation=30
)

plt.ylabel("Score")

plt.title("Confianza de las respuestas")

plt.show()

# Discusión

Observe cómo disminuye la confianza conforme aumenta el número de respuestas.

Generalmente:

- la primera respuesta posee la mayor probabilidad.
- las siguientes representan alternativas menos probables.

# Preguntas sin respuesta

Uno de los principales desafíos consiste en responder preguntas cuya respuesta **no aparece dentro del contexto**.

In [24]:
# =====================================================
# CONTEXTO
# =====================================================

contexto = """
Python fue creado por Guido van Rossum y apareció en 1991.

Actualmente es uno de los lenguajes más utilizados para Ciencia de Datos.
"""

In [25]:
preguntas = [

"¿Quién creó Python?",

"¿En qué año apareció?",

"¿Cuál es la capital de Francia?",

"¿Quién creó Java?"
]

for pregunta in preguntas:

    print("="*70)

    print(pregunta)

    respuestas = responder_pregunta(
        contexto,
        pregunta,
        top_k=3
    )

    for r in respuestas:

        print(r)

¿Quién creó Python?
{'answer': 'guido van rossum', 'score': 0.969997227191925, 'start': 17, 'end': 22}
{'answer': 'van rossum', 'score': 0.011163368821144104, 'start': 19, 'end': 22}
{'answer': 'guido van rossum y', 'score': 0.005150328855961561, 'start': 17, 'end': 23}
¿En qué año apareció?
{'answer': '1991', 'score': 0.8395678997039795, 'start': 26, 'end': 26}
{'answer': '[CLS] ¿ en que ano aparecio? [SEP] python fue creado por guido van rossum y aparecio en 1991', 'score': 0.06378805637359619, 'start': 0, 'end': 26}
{'answer': '1991.', 'score': 0.0019234357168897986, 'start': 26, 'end': 27}
¿Cuál es la capital de Francia?
{'answer': '[CLS] ¿ cual es la capital de francia? [SEP] python fue creado por guido van rossum y aparecio en 1991', 'score': 0.001322624972090125, 'start': 0, 'end': 28}
{'answer': '[CLS] ¿ cual es la capital de francia', 'score': 0.0009502682951278985, 'start': 0, 'end': 8}
{'answer': '[CLS] ¿ cual es la capital de francia? [SEP] python', 'score': 0.0006850895588

# Análisis

Observe cuidadosamente las respuestas.

Preguntas:

- ¿El modelo inventó información?
- ¿Intentó responder aunque el contexto no contiene la respuesta?
- ¿Qué ocurre con el score?

In [26]:
# =====================================================
# DETECTAR BAJA CONFIANZA
# =====================================================

def responder_seguro(contexto, pregunta, umbral=0.40):

    respuesta = responder_pregunta(
        contexto,
        pregunta,
        top_k=1
    )[0]

    if respuesta["score"] < umbral:

        print("⚠️ Confianza insuficiente.")

        print("Se recomienda revisar manualmente.")

    else:

        print("Respuesta:", respuesta["answer"])

        print("Score:", round(respuesta["score"],4))

In [27]:
responder_seguro(
    contexto,
    "¿Quién creó Java?"
)

⚠️ Confianza insuficiente.
Se recomienda revisar manualmente.


# Buenas prácticas

Para obtener mejores resultados:

- utilizar contextos claros.
- evitar información redundante.
- formular preguntas específicas.
- mantener el contexto relacionado con la pregunta.
- evitar documentos excesivamente largos.

# Actividad de laboratorio

Cada estudiante deberá construir un contexto de aproximadamente 300 palabras relacionado con un tema de su especialidad.

Posteriormente deberá formular diez preguntas y registrar:

- respuesta
- score
- observaciones

Finalmente analizará cuáles preguntas fueron respondidas correctamente y cuáles presentaron dificultades.

# Ejercicio de reflexión

Explique:

1. ¿Qué representa el score?

2. ¿Qué ventajas tiene utilizar `top_k`?

3. ¿Por qué un modelo extractivo no puede responder preguntas cuya información no aparece en el contexto?

4. ¿En qué aplicaciones reales utilizaría Question Answering?

# Resumen del bloque

En este bloque aprendimos a:

- Analizar múltiples respuestas mediante `top_k`.
- Interpretar el nivel de confianza (`score`).
- Visualizar las respuestas utilizando tablas y gráficos.
- Detectar preguntas cuya respuesta no aparece en el contexto.
- Implementar un mecanismo simple para filtrar respuestas de baja confianza.
- Aplicar buenas prácticas para mejorar el rendimiento de los sistemas de Question Answering.

En el siguiente bloque se compararán distintos modelos de QA en español, se evaluará su desempeño con los mismos contextos y se analizarán sus fortalezas y limitaciones.

# Comparación de Modelos de Question Answering

Hasta este momento hemos utilizado un único modelo.

Sin embargo, Hugging Face ofrece numerosos modelos de QA entrenados para diferentes idiomas y dominios.

En esta práctica compararemos distintos modelos utilizando exactamente el mismo contexto y las mismas preguntas.

Analizaremos:

- calidad de la respuesta
- score de confianza
- velocidad
- ventajas
- limitaciones

In [ ]:
# =====================================================
# MODELOS A COMPARAR
# =====================================================
# Se reutiliza el diccionario MODELOS definido en el Bloque 2,
# ya verificado contra el Hub de Hugging Face.

MODELOS

# Contexto de evaluación

Todos los modelos responderán utilizando exactamente el mismo contexto.

De esta forma la comparación será objetiva.

In [29]:
contexto = """
La Universidad de El Salvador fue fundada el 16 de febrero de 1841.

Es la universidad pública más antigua del país.

Cuenta con varias facultades distribuidas en diferentes sedes.

Su misión consiste en contribuir al desarrollo científico, tecnológico y cultural de El Salvador mediante la docencia, investigación y proyección social.
"""

In [30]:
preguntas = [

"¿Cuándo fue fundada la Universidad de El Salvador?",

"¿Qué tipo de universidad es?",

"¿Cuál es su misión?"

]

# Función para cargar un modelo

Crearemos una función que facilite la carga dinámica de diferentes modelos.

In [ ]:
# =====================================================
# CARGA DINÁMICA DE MODELOS (con caché)
# =====================================================
# Reutiliza la clase ModeloQA del Bloque 2. El caché evita volver a
# descargar y reinstanciar un modelo ya cargado, lo cual distorsionaría
# la medición de tiempos de inferencia.

_cache_modelos = {}

def cargar_modelo(etiqueta):
    """Devuelve el ModeloQA correspondiente a una etiqueta de MODELOS."""
    if etiqueta not in _cache_modelos:
        print(f"Descargando y cargando: {etiqueta} ...")
        _cache_modelos[etiqueta] = ModeloQA(MODELOS[etiqueta], etiqueta=etiqueta)
    return _cache_modelos[etiqueta]


print("Modelos disponibles:", list(MODELOS))

In [ ]:
qa_principal = cargar_modelo(MODELO_PRINCIPAL)

pd.DataFrame([qa_principal.describir()]).T.rename(columns={0: "Valor"})

In [ ]:
for pregunta in preguntas:

    print("=" * 70)
    print(pregunta)

    resultado = qa_principal.responder(contexto, pregunta, top_k=1)[0]

    print(f"Respuesta : {resultado['answer']}")
    print(f"Score     : {resultado['score']:.4f}")

# Midiendo el tiempo de respuesta

Además de la calidad de la respuesta, también es importante medir el tiempo de inferencia.

In [ ]:
respuestas, segundos = qa_principal.responder_con_tiempo(
    contexto,
    "¿Cuál es la misión de la universidad?"
)

print("Tiempo:", round(segundos, 4), "segundos")
print(respuestas[0])

# Comparación automática

Ahora evaluaremos todos los modelos.

In [ ]:
# =====================================================
# VERIFICACIÓN DE DISPONIBILIDAD EN HUGGING FACE
# =====================================================
# Se comprueba que cada identificador exista en el Hub antes de
# intentar descargarlo. Evita fallos a mitad de la comparación.

from huggingface_hub import model_info

verificacion = []

for etiqueta, ruta in MODELOS.items():
    try:
        info = model_info(ruta)
        verificacion.append({
            "Etiqueta": etiqueta,
            "Modelo": ruta,
            "Estado": "Disponible",
            "Arquitectura": info.config.get("architectures", ["?"])[0],
            "Descargas": info.downloads,
        })
    except Exception as e:
        verificacion.append({
            "Etiqueta": etiqueta,
            "Modelo": ruta,
            "Estado": f"NO DISPONIBLE ({type(e).__name__})",
            "Arquitectura": "-",
            "Descargas": "-",
        })

pd.DataFrame(verificacion)

In [ ]:
# =====================================================
# COMPARACIÓN AUTOMÁTICA DE LOS TRES MODELOS
# =====================================================

pregunta_eval = "¿Cuándo fue fundada la Universidad de El Salvador?"

resultados = []

for etiqueta in MODELOS:

    print("=" * 70)
    print(etiqueta)

    modelo = cargar_modelo(etiqueta)

    respuestas, segundos = modelo.responder_con_tiempo(
        contexto,
        pregunta_eval,
        top_k=1
    )

    resultados.append({
        "Modelo": etiqueta,
        "Respuesta": respuestas[0]["answer"],
        "Score": respuestas[0]["score"],
        "Tiempo": segundos,
        "Parámetros (M)": round(modelo.n_parametros / 1e6, 1),
    })

print("\nEvaluación terminada.")

In [ ]:
tabla_modelos = pd.DataFrame(resultados)

tabla_modelos.to_csv("results/comparacion_modelos.csv", index=False)

tabla_modelos

In [ ]:
tabla_modelos.sort_values(

    by="Score",

    ascending=False
)

# Analizando los resultados

Observe:

- ¿Todos responden igual?
- ¿Cuál obtuvo mayor score?
- ¿Cuál fue más rápido?
- ¿Existe relación entre velocidad y precisión?

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(tabla_modelos["Modelo"], tabla_modelos["Score"], color="#4C72B0")
ax.set_ylabel("Score")
ax.set_title("Comparación de confianza entre modelos")
ax.tick_params(axis="x", rotation=15)

for i, v in enumerate(tabla_modelos["Score"]):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")

fig.tight_layout()
fig.savefig("results/graficos/comparacion_scores.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(tabla_modelos["Modelo"], tabla_modelos["Tiempo"], color="#DD8452")
ax.set_ylabel("Segundos")
ax.set_title("Tiempo de inferencia por modelo")
ax.tick_params(axis="x", rotation=15)

for i, v in enumerate(tabla_modelos["Tiempo"]):
    ax.text(i, v, f"{v:.3f}s", ha="center", va="bottom")

fig.tight_layout()
fig.savefig("results/graficos/comparacion_tiempos.png", dpi=150)
plt.show()

# Ventajas de RoBERTa

- Excelente precisión.
- Muy buen rendimiento en español.
- Robusto frente a preguntas complejas.
- Buena generalización.

# Ventajas de BETO

- Entrenado específicamente para español.
- Excelente tokenización.
- Muy útil para documentos administrativos y académicos.

# Limitaciones observadas

Los modelos extractivos presentan algunas limitaciones:

- No generan texto nuevo.
- No realizan razonamiento complejo.
- No combinan información de múltiples documentos.
- Solo extraen fragmentos del contexto.

# Actividad 1

Repita la comparación utilizando un contexto relacionado con:

- Bases de Datos
- Inteligencia Artificial
- Redes Neuronales
- Ciencia de Datos

Analice si el mejor modelo continúa siendo el mismo.

# Actividad 2

Seleccione un artículo de Wikipedia en español.

Construya cinco preguntas.

Compare las respuestas utilizando los tres modelos.

# Ejercicio

Complete la siguiente tabla.

|Modelo|Ventajas|Desventajas|
|--------|----------|-------------|
|RoBERTa-BNE|||
|BETO-XQuAD|||
|BETO-SQuAD|||

# Preguntas de reflexión

1. ¿Qué modelo obtuvo la mayor confianza?

2. ¿Cuál respondió más rápido?

3. ¿Cuál recomendaría para documentos en español?

4. ¿Qué características debe tener un buen modelo QA?

# Resumen del Bloque

En este bloque aprendimos a:

- Comparar distintos modelos de Question Answering.
- Medir el tiempo de inferencia.
- Analizar el nivel de confianza.
- Evaluar precisión y velocidad.
- Interpretar las fortalezas y limitaciones de cada modelo.

En el siguiente bloque construiremos un **proyecto integrador**, desarrollando un sistema de Question Answering sobre documentos extensos y aplicando técnicas para manejar contextos que exceden la longitud máxima admitida por los modelos Transformer.

# Proyecto

Hasta este momento hemos utilizado pequeños contextos.

Sin embargo, en aplicaciones reales los documentos pueden contener cientos o miles de palabras.

Por ejemplo:

- Reglamentos universitarios
- Manuales técnicos
- Artículos científicos
- Leyes
- Normativas ISO
- Libros
- Historias clínicas

En este laboratorio construiremos un sistema capaz de responder preguntas sobre documentos extensos.

# Problema

Los modelos Transformer poseen una longitud máxima de entrada.

En la mayoría de modelos BERT y RoBERTa el límite es aproximadamente:

- 512 tokens

Por ello un documento largo no puede enviarse directamente al modelo.

La solución consiste en dividir el documento en pequeños fragmentos.

# Estrategia

Nuestro sistema seguirá el siguiente flujo:

Documento
      │
      ▼
Dividir en fragmentos
      │
      ▼
Pregunta
      │
      ▼
QA sobre cada fragmento
      │
      ▼
Comparar scores
      │
      ▼
Seleccionar la mejor respuesta

In [42]:
# =====================================================
# DOCUMENTO DE EJEMPLO
# =====================================================

documento = """
La Universidad de El Salvador fue fundada el 16 de febrero de 1841.

Actualmente posee varias facultades distribuidas en diferentes sedes.

Su misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.

La Facultad de Ingeniería y Arquitectura ofrece diversas carreras de ingeniería.

La Escuela de Ingeniería de Sistemas Informáticos forma profesionales especializados en desarrollo de software, inteligencia artificial, bases de datos, ciencia de datos y redes computacionales.

Además desarrolla proyectos de investigación y vinculación con la sociedad.
""" * 20

# Observación

En este ejemplo repetimos el texto varias veces para simular un documento extenso.

In [43]:
len(documento)

12460

In [44]:
print(documento[:800])


La Universidad de El Salvador fue fundada el 16 de febrero de 1841.

Actualmente posee varias facultades distribuidas en diferentes sedes.

Su misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.

La Facultad de Ingeniería y Arquitectura ofrece diversas carreras de ingeniería.

La Escuela de Ingeniería de Sistemas Informáticos forma profesionales especializados en desarrollo de software, inteligencia artificial, bases de datos, ciencia de datos y redes computacionales.

Además desarrolla proyectos de investigación y vinculación con la sociedad.

La Universidad de El Salvador fue fundada el 16 de febrero de 1841.

Actualmente posee varias facultades distribuidas en diferentes sedes.

Su misión consiste en formar profesi


# Dividir el documento

Ahora construiremos una función que divida el documento en fragmentos.

In [45]:
# =====================================================
# DIVIDIR DOCUMENTO
# =====================================================

def dividir_texto(texto, longitud=700):

    fragmentos = []

    inicio = 0

    while inicio < len(texto):

        fin = inicio + longitud

        fragmentos.append(texto[inicio:fin])

        inicio += longitud

    return fragmentos

In [46]:
fragmentos = dividir_texto(documento)

print("Número de fragmentos:", len(fragmentos))

Número de fragmentos: 18


In [47]:
for i, f in enumerate(fragmentos):

    print("="*60)

    print("Fragmento", i+1)

    print(f[:250])

Fragmento 1

La Universidad de El Salvador fue fundada el 16 de febrero de 1841.

Actualmente posee varias facultades distribuidas en diferentes sedes.

Su misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y p
Fragmento 2
ente posee varias facultades distribuidas en diferentes sedes.

Su misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.

La Facultad de Ingeniería y Arquitectura ofrece diversas c
Fragmento 3
siste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.

La Facultad de Ingeniería y Arquitectura ofrece diversas carreras de ingeniería.

La Escuela de Ingeniería de Sistemas Informáticos for
Fragmento 4
, investigación y proyección social.

La Facultad de Ingeniería y Arquitectura ofrece diversas carreras de ingeniería.

La Escuela de Ingeniería de Sistemas Informáticos forma profesionales especiali

# Aplicando Question Answering a cada fragmento

La pregunta será enviada a todos los fragmentos.

Posteriormente compararemos los scores.

In [48]:
pregunta = "¿Cuál es la misión de la Universidad de El Salvador?"

In [50]:
respuestas = []

for i, fragmento in enumerate(fragmentos):

    resultado = responder_pregunta(
        fragmento,
        pregunta,
        top_k=1
    )[0]

    resultado["fragmento"] = i + 1

    respuestas.append(resultado)

In [ ]:
tabla_fragmentos = pd.DataFrame(respuestas)

tabla_fragmentos

In [ ]:
tabla_fragmentos.sort_values(
    by="score",
    ascending=False
)

# Mejor respuesta

Seleccionaremos automáticamente la respuesta con mayor score.

In [ ]:
mejor = tabla_fragmentos.sort_values(
    by="score",
    ascending=False
).iloc[0]

mejor

In [54]:
print("="*60)

print("Pregunta")

print(pregunta)

print()

print("Respuesta")

print(mejor["answer"])

print()

print("Score")

print(round(mejor["score"],4))

print()

print("Fragmento")

print(mejor["fragmento"])

Pregunta
¿Cuál es la misión de la Universidad de El Salvador?

Respuesta
formar profesionales con alto compromiso social mediante la docencia, investigacion y proyeccion social

Score
0.5282

Fragmento
15


# Visualización de resultados

Representaremos gráficamente la confianza obtenida en cada fragmento.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.plot(
    tabla_fragmentos["fragmento"],
    tabla_fragmentos["score"],
    marker="o"
)

plt.xlabel("Fragmento")

plt.ylabel("Score")

plt.title("Confianza obtenida por fragmento")

plt.grid(True)

plt.show()

# Mejorando el sistema

El algoritmo anterior puede mejorarse mediante:

- solapamiento entre fragmentos (*overlap*)
- búsqueda semántica
- embeddings
- recuperación de documentos
- reranking

In [56]:
def dividir_con_solapamiento(
    texto,
    longitud=700,
    overlap=150
):

    fragmentos = []

    inicio = 0

    while inicio < len(texto):

        fin = inicio + longitud

        fragmentos.append(texto[inicio:fin])

        inicio += longitud - overlap

    return fragmentos

In [57]:
fragmentos = dividir_con_solapamiento(documento)

print(len(fragmentos))

23


# ¿Por qué utilizar solapamiento?

En ocasiones una respuesta queda dividida entre dos fragmentos.

El solapamiento reduce la probabilidad de perder información importante.

# Aplicaciones reales

Los sistemas QA sobre documentos extensos son utilizados en:

- asistentes jurídicos
- motores de búsqueda
- universidades
- hospitales
- bancos
- aseguradoras
- soporte técnico
- bibliotecas digitales

# Mini Proyecto

Desarrolle un asistente capaz de responder preguntas sobre un documento de su elección.

Puede utilizar:

- Reglamento General de la UES.
- Manual de usuario.
- Artículo científico.
- Norma ISO.
- Documento técnico.

# Actividad Individual

1. Seleccione un documento de al menos cinco páginas.
2. Divídalo en fragmentos.
3. Formule diez preguntas.
4. Registre las respuestas y los scores.
5. Analice los errores encontrados.

# Actividad en Equipos

Cada equipo comparará:

- tamaño del fragmento
- uso o no de solapamiento
- tiempo de ejecución
- precisión de las respuestas

Posteriormente elaborará un informe con sus conclusiones.

# Preguntas de Reflexión

1. ¿Por qué es necesario dividir documentos largos?
2. ¿Qué ventajas ofrece el solapamiento?
3. ¿Cómo seleccionar la mejor respuesta?
4. ¿Qué limitaciones presenta este enfoque?
5. ¿Cómo podría mejorarse utilizando búsqueda semántica?

# Conclusiones del Bloque

En este laboratorio se construyó un sistema básico de Question Answering sobre documentos extensos.

Se implementaron técnicas de:

- fragmentación de documentos (*chunking*),
- evaluación de múltiples respuestas,
- selección automática de la respuesta con mayor confianza,
- uso de solapamiento entre fragmentos.

Estos conceptos constituyen la base de arquitecturas modernas como **Retrieval-Augmented Generation (RAG)**, donde un sistema recupera información relevante antes de generar una respuesta. En el siguiente bloque se explorará cómo combinar recuperación de información y modelos de lenguaje para construir asistentes conversacionales sobre colecciones de documentos.

# Proyecto Final

## ¿Qué es RAG?

RAG (Retrieval-Augmented Generation) es una arquitectura moderna que combina:

- Recuperación de información (Information Retrieval)
- Modelos de lenguaje (LLM)
- Embeddings
- Búsqueda semántica

En lugar de enviar un documento completo al modelo, primero se recuperan únicamente los fragmentos más relevantes.

# ¿Por qué utilizar RAG?

Suponga un documento de 500 páginas.

Un modelo BERT solo admite aproximadamente 512 tokens.

La solución consiste en:

1. Dividir el documento.
2. Convertir cada fragmento en un embedding.
3. Buscar los fragmentos más parecidos a la pregunta.
4. Ejecutar Question Answering únicamente sobre esos fragmentos.

Esta estrategia mejora significativamente la precisión y reduce el tiempo de respuesta.

# Arquitectura General

```
Documento
      │
      ▼
Fragmentación (Chunking)
      │
      ▼
Embeddings
      │
      ▼
Base Vectorial
      │
      ▼
Pregunta
      │
      ▼
Embedding de la pregunta
      │
      ▼
Búsqueda Semántica
      │
      ▼
Top-K fragmentos
      │
      ▼
Question Answering
      │
      ▼
Respuesta
```

In [58]:
# =====================================================
# INSTALAR LIBRERÍAS
# =====================================================

!pip install -q sentence-transformers
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.6 MB/s eta 0:00:00


In [59]:
# =====================================================
# IMPORTAR LIBRERÍAS
# =====================================================

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

# Modelo de Embeddings

Los modelos de embeddings convierten un texto en un vector numérico.

Estos vectores representan el significado semántico del texto.

En este laboratorio utilizaremos:

```
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

Este modelo soporta múltiples idiomas, incluido el español.

In [60]:
modelo_embeddings = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [61]:
fragmentos = dividir_con_solapamiento(
    documento,
    longitud=600,
    overlap=120
)

print("Fragmentos:", len(fragmentos))

Fragmentos: 26


In [62]:
embeddings = modelo_embeddings.encode(
    fragmentos,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(26, 384)


# Base Vectorial

Construiremos una base vectorial utilizando FAISS.

FAISS fue desarrollado por Meta AI y permite realizar búsquedas muy rápidas sobre millones de vectores.

In [63]:
dimension = embeddings.shape[1]

indice = faiss.IndexFlatL2(dimension)

indice.add(embeddings)

print("Vectores almacenados:", indice.ntotal)

Vectores almacenados: 26


In [64]:
pregunta = "¿Cuál es la misión de la Universidad de El Salvador?"

embedding_pregunta = modelo_embeddings.encode(
    [pregunta],
    convert_to_numpy=True
)

In [65]:
k = 3

distancias, indices = indice.search(
    embedding_pregunta,
    k
)

indices

array([[14, 18,  5]])

In [66]:
fragmentos_recuperados = []

for indice_fragmento in indices[0]:

    fragmentos_recuperados.append(
        fragmentos[indice_fragmento]
    )

fragmentos_recuperados

['ses de datos, ciencia de datos y redes computacionales.\n\nAdemás desarrolla proyectos de investigación y vinculación con la sociedad.\n\nLa Universidad de El Salvador fue fundada el 16 de febrero de 1841.\n\nActualmente posee varias facultades distribuidas en diferentes sedes.\n\nSu misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.\n\nLa Facultad de Ingeniería y Arquitectura ofrece diversas carreras de ingeniería.\n\nLa Escuela de Ingeniería de Sistemas Informáticos forma profesionales especializados en desarrollo de software, inte',
 'les.\n\nAdemás desarrolla proyectos de investigación y vinculación con la sociedad.\n\nLa Universidad de El Salvador fue fundada el 16 de febrero de 1841.\n\nActualmente posee varias facultades distribuidas en diferentes sedes.\n\nSu misión consiste en formar profesionales con alto compromiso social mediante la docencia, investigación y proyección social.\n\nLa Facultad de Ingen

# Recuperación Semántica

Observe que los fragmentos recuperados contienen información relacionada con la pregunta, incluso cuando las palabras utilizadas no son exactamente las mismas.

Esta es una de las principales ventajas de los embeddings.

In [67]:
mejor_respuesta = None

for fragmento in fragmentos_recuperados:

    respuesta = responder_pregunta(
        fragmento,
        pregunta,
        top_k=1
    )[0]

    if (mejor_respuesta is None or
        respuesta["score"] > mejor_respuesta["score"]):

        mejor_respuesta = respuesta

mejor_respuesta

{'answer': 'formar profesionales con alto compromiso social mediante la docencia, investigacion y proyeccion social',
 'score': 0.49557724595069885,
 'start': 69,
 'end': 84}

In [68]:
print("="*60)

print("Pregunta")

print(pregunta)

print()

print("Respuesta")

print(mejor_respuesta["answer"])

print()

print("Confianza")

print(round(mejor_respuesta["score"],4))

Pregunta
¿Cuál es la misión de la Universidad de El Salvador?

Respuesta
formar profesionales con alto compromiso social mediante la docencia, investigacion y proyeccion social

Confianza
0.4956


# Comparación

## QA tradicional

Documento completo

↓

Modelo QA

↓

Respuesta

---

## RAG

Documento

↓

Fragmentación

↓

Embeddings

↓

Búsqueda Semántica

↓

Question Answering

↓

Respuesta

In [69]:
import pandas as pd

comparacion = pd.DataFrame({

    "Característica":[

        "Escalabilidad",

        "Velocidad",

        "Documentos grandes",

        "Precisión",

        "Uso de memoria"

    ],

    "QA Tradicional":[

        "Baja",

        "Media",

        "Limitado",

        "Media",

        "Alta"

    ],

    "RAG":[

        "Muy alta",

        "Alta",

        "Excelente",

        "Alta",

        "Media"

    ]

})

comparacion

,Característica,QA Tradicional,RAG
0,Escalabilidad,Baja,Muy alta
1,Velocidad,Media,Alta
2,Documentos grandes,Limitado,Excelente
3,Precisión,Media,Alta
4,Uso de memoria,Alta,Media


# Aplicaciones Reales

Los sistemas RAG son utilizados en:

- ChatGPT Enterprise
- Microsoft Copilot
- Gemini
- Claude
- Asistentes jurídicos
- Bibliotecas digitales
- Hospitales
- Universidades
- Bancos
- Empresas de soporte técnico

# Actividad de Laboratorio

Seleccione un documento técnico de al menos 20 páginas.

Implemente un sistema RAG que permita responder preguntas sobre dicho documento.

Registre:

- número de fragmentos
- tiempo de indexación
- tiempo de búsqueda
- score de las respuestas
- observaciones

# Proyecto Final

Desarrolle un asistente inteligente para la Universidad de El Salvador que pueda responder preguntas sobre:

- Reglamento General
- Reglamento Académico
- Normativa de Graduación
- Manuales administrativos
- Guías estudiantiles

El sistema deberá utilizar:

- Embeddings
- FAISS
- Question Answering
- Búsqueda semántica
- Recuperación Top-K

# Ejercicios Propuestos

1. Cambie el modelo de embeddings y compare resultados.
2. Experimente con diferentes tamaños de fragmento (300, 500 y 800 caracteres).
3. Evalúe el efecto del solapamiento (`overlap`) en la calidad de las respuestas.
4. Pruebe diferentes valores de `k` en la búsqueda semántica.
5. Analice el impacto del tamaño de la base vectorial en el tiempo de búsqueda.

# Conclusiones

En este bloque se desarrolló un sistema básico de Retrieval-Augmented Generation (RAG), integrando:

- fragmentación de documentos,
- generación de embeddings,
- indexación con FAISS,
- búsqueda semántica,
- recuperación Top-K,
- Question Answering extractivo.

Este enfoque supera las limitaciones de los modelos QA tradicionales al permitir trabajar con documentos extensos y seleccionar únicamente los fragmentos más relevantes antes de responder. Constituye la base de muchos asistentes inteligentes modernos y representa una de las arquitecturas más utilizadas para construir sistemas de consulta sobre documentos empresariales, académicos y científicos.